# Chest X-ray — FULL run (all 7 models)

**Run only AFTER colab_small.ipynb returned GO.** Trains all 7 models at full epochs and produces the final figures/tables. Same entrypoint: `scripts/run_pipeline.py`.

Cells 1-6 are identical to the small run (setup + path check).

## Cell 1 — mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Cell 2 — clone code from GitHub

In [ ]:
import os, subprocess
REPO = 'https://github.com/kamlishgoswami/Chest_Xray.git'
if not os.path.exists('/content/Chest_Xray'):
    subprocess.run(['git','clone', REPO, '/content/Chest_Xray'], check=True)
else:
    subprocess.run(['git','-C','/content/Chest_Xray','pull'], check=True)
os.chdir('/content/Chest_Xray'); print('cwd:', os.getcwd())

## Cell 3 — unzip data to fast local disk (slow, once per session)

In [ ]:
import os, glob, subprocess
DRIVE_DIR = '/content/drive/MyDrive/cxr_data'
os.makedirs('data/raw', exist_ok=True)
for z in glob.glob(f'{DRIVE_DIR}/*.zip'):
    print('unzipping', os.path.basename(z))
    subprocess.run(['unzip','-q','-o', z, '-d','data/raw'], check=True)
print('folders:', os.listdir('data/raw'))

## Cell 4 — bring the manifest

In [ ]:
import os, shutil, subprocess
DRIVE_DIR = '/content/drive/MyDrive/cxr_data'
if os.path.exists(f'{DRIVE_DIR}/manifest.csv'):
    shutil.copy(f'{DRIVE_DIR}/manifest.csv','data/manifest.csv'); print('manifest copied')
else:
    subprocess.run(['python','scripts/build_manifest.py'], check=True); print('rebuilt')

## Cell 5 — install deps + GPU check (GPU must NOT be empty)

In [ ]:
import subprocess
subprocess.run(['pip','-q','install','-r','requirements.txt'], check=True)
import tensorflow as tf
print('GPU:', tf.config.list_physical_devices('GPU'))

## Cell 6 — DATA PATH CHECK ⛔ (stop if not 0)

In [ ]:
import pandas as pd, os
df = pd.read_csv('data/manifest.csv')
missing = [p for p in df['image_path'].head(300) if not os.path.exists(p)]
print('missing:', len(missing), '/ 300')
if missing:
    print('EXAMPLE:', missing[0]); print('folders:', os.listdir('data/raw'))
    print('>>> STOP: fix folder mapping before running.')
else:
    print('>>> OK: paths resolve. Continue.')

## Cell 7 — FULL pipeline on all 7 models (long: hours on T4)
`--resume` skips any model already trained, so this is safe to re-run after a disconnect.

In [ ]:
import subprocess, sys
MODELS = ['densenet201','efficientnetb0','resnet50','mobilenetv3large','xception','vit','lenet5']
subprocess.run([sys.executable,'scripts/run_pipeline.py',
                '--models',*MODELS,'--epochs','100','--batch-size','64'], check=True)

## Cell 8 — final C3 coupling (all 7 models)

In [ ]:
import json
print(json.dumps(json.load(open('results/c3_coupling.json')), indent=2))

## Cell 9 — list final outputs (figures + tables for the paper)

In [ ]:
import os
print('FIGURES:', os.listdir('results/figures'))
print('TABLES :', os.listdir('results/tables'))
print('STATS  :', os.path.exists('results/stats_summary.json'))

## Cell 10 — SAVE everything to Drive (do not skip)

In [ ]:
import subprocess, datetime
stamp = datetime.date.today().isoformat()
subprocess.run(['cp','-r','results',f'/content/drive/MyDrive/cxr_data/results_full_{stamp}'], check=True)
print('saved -> Drive/cxr_data/results_full_'+stamp)